# 06 — Evolving bands, retests and unbroken-age strength

Use the first 200 trades for the initial explanation, then update discovery after every subsequent observed event. Bands exclude valley plateaus. A broken band waits for an opposite-side retest before changing role. Each role period has its own age. The event slider shows only history available at that cutoff.


In [ ]:
# Basic setup. Generated data and charts must stay outside the repository.
import os
import sys
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import subprocess
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path("D:/TradingCodes/quant-research-workbench")
RUNTIME_ROOT = Path("D:/TradingML/runtimes")
SOURCE_RUN = RUNTIME_ROOT / "level-book-v8/reaction-demo/20261020T204419563339Z"
EXPECTED_SOURCE_HASH = "b40270ba8fc4055d515fa9061ce5333cce152890de23e249dbbc827e7e0ec70d"
N_EVENTS = 200  # Discovery prefix only; the replay uses all 2,000 saved events.
DISCOVERY_EVENT = N_EVENTS
# Explicit illustrative settings, fixed before inspecting the result.
TAU_SECONDS = 1.0          # Same 1-second reference used in our discussion; not a candle timeframe.
SCORE_THRESHOLD = 0.60     # A display/selection rule, not a calibrated confidence threshold.
WEIGHT_COUNT = 1.0 / 3     # Equal weights reproduce (normalized count + volume + time) / 3.
WEIGHT_VOLUME = 1.0 / 3
WEIGHT_TIME = 1.0 / 3
assert TAU_SECONDS > 0
assert 0 < SCORE_THRESHOLD <= 1
assert min(WEIGHT_COUNT, WEIGHT_VOLUME, WEIGHT_TIME) >= 0
assert np.isclose(WEIGHT_COUNT + WEIGHT_VOLUME + WEIGHT_TIME, 1.0)
assert REPO.is_dir() and RUNTIME_ROOT.is_dir(), "Required root unavailable"
RUN = RUNTIME_ROOT / "level-book-v8/unbroken-age-bands" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True, exist_ok=False)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(16)
print("Output directory:", RUN)


## 1. Load and verify the original sample
This reads the saved canonical-derived snapshot, not raw SIP flatfiles. We validate its original content hash before taking the requested prefix. If it is missing or changed, stop rather than silently fetching a different sample.

The original extraction excluded delayed/unknown-clock and condition-ineligible trades. Its audit and provenance are retained. Price is stored in exact units of 1/10,000 dollar. Ordinal is metadata, not a predictor: order remains `(SIP timestamp, ordinal)`, exactly as in notebook 01. Time differences can be zero for equal timestamps.

We slice **before** calculating shifts. The first event outside the prefix must not reveal the final event's departure.

In [ ]:
# Recreate the original loader's hashed records, in their original order.
source = pl.read_parquet(SOURCE_RUN / "source_trades.parquet")
provenance = json.loads((SOURCE_RUN / "source_provenance.json").read_text(encoding="utf-8"))
source_records = source.select("ordinal", "time_us", "price_units", "volume").to_dicts()
# ClickHouse JSON encoded integral sizes as integers; Parquet promoted the
# mixed size column to Float64. Restore only that JSON representation for hashing.
for record in source_records:
    if float(record["volume"]).is_integer():
        record["volume"] = int(record["volume"])
source_hash = hashlib.sha256(json.dumps(source_records, sort_keys=True).encode()).hexdigest()
assert source_hash == provenance["sample_sha256"] == EXPECTED_SOURCE_HASH
assert source.height == 2000 and provenance["ticker"] == "AAPL"
assert provenance["day"] == "2026-08-20"
assert 3 <= N_EVENTS <= source.height
assert source["ordinal"].n_unique() == source.height
assert source.select("time_us", "ordinal").equals(source.sort("time_us", "ordinal").select("time_us", "ordinal"))
raw = source.head(N_EVENTS)
# These are the three original quantities; event and ordinal identify rows.
display(raw.select("event", "ordinal", "time", "price", "volume").head(15))
print("Selected events:", raw.height, "unique prices:", raw["price_units"].n_unique())
print("Time span:", raw["time"][0], "to", raw["time"][-1])
display(pl.DataFrame(provenance["audit"]))
# Explicit regular-session origin, not the first sampled trade or local machine time.
from zoneinfo import ZoneInfo
session_open = datetime.fromisoformat(provenance["day"]+"T09:30:00").replace(tzinfo=ZoneInfo("America/New_York"))
SESSION_OPEN_US = int(session_open.timestamp()*1_000_000)
assert source["time_us"].min() >= SESSION_OPEN_US
print("Session origin:",session_open.isoformat(),"UTC:",session_open.astimezone(timezone.utc).isoformat())


## 2. Our agreed step one: six vectorized differences
For any column x:

- backward = current minus previous;
- forward = next minus current.

Time differences are seconds, price differences dollars, and volume differences changes in individual trade size. Arrival and departure are the signs of the respective price differences. A zero price difference means unchanged, not missing.

The first backward values and final forward values remain null. Forward time is the current trade's completed hold contribution. We do not add elapsed time for the pending final row or pretend its reaction is unchanged.
Additional vectorized fields: session elapsed seconds, observed-sample high/low, and running low/high starting at discovery. Sample extremes are not certified full-session extremes because this is a bounded trade sample. Running discovery extremes support first-break detection without comparing every band against every trade. Equal timestamps remain ordered by canonical ordinal; they add no artificial elapsed time.


In [ ]:
def step_one(frame):
    """Derive adjacent-row differences on precisely the supplied prefix."""
    return frame.with_columns(
        # These time/extreme fields depend only on the current and previous rows.
        ((pl.col("time_us")-SESSION_OPEN_US)/1e6).alias("session_elapsed_seconds"),
        pl.col("price_units").cum_min().alias("sample_low_units"),
        pl.col("price_units").cum_max().alias("sample_high_units"),
        # Null before discovery: earlier extremes must not break a newly established band.
        pl.when(pl.col("event")>=DISCOVERY_EVENT).then(pl.col("price_units")).otherwise(None)
          .cum_min().alias("since_discovery_low_units"),
        pl.when(pl.col("event")>=DISCOVERY_EVENT).then(pl.col("price_units")).otherwise(None)
          .cum_max().alias("since_discovery_high_units"),
        ((pl.col("time_us") - pl.col("time_us").shift(1)) / 1e6).alias("time_backward"),
        ((pl.col("time_us").shift(-1) - pl.col("time_us")) / 1e6).alias("time_forward"),
        ((pl.col("price_units") - pl.col("price_units").shift(1)) / 10000).alias("price_backward"),
        ((pl.col("price_units").shift(-1) - pl.col("price_units")) / 10000).alias("price_forward"),
        (pl.col("volume") - pl.col("volume").shift(1)).alias("volume_backward"),
        (pl.col("volume").shift(-1) - pl.col("volume")).alias("volume_forward"),
    ).with_columns(
        pl.col("price_backward").sign().cast(pl.Int8).alias("arrival"),
        pl.col("price_forward").sign().cast(pl.Int8).alias("departure"),
    )

step1 = step_one(raw)
step_columns = ["event", "time", "price", "volume", "time_backward", "time_forward",
                "price_backward", "price_forward", "volume_backward", "volume_forward"]
display(step1.select(step_columns).head(15))
display(step1.select(step_columns).tail(3))
display(step1.select("event","time","price","session_elapsed_seconds","sample_low_units",
                    "sample_high_units","since_discovery_low_units","since_discovery_high_units").tail(5))


## 3. Group by exact price and the nine direction pairs

| Arrival / departure | Down (−1) | Unchanged (0) | Up (+1) |
|---|---|---|---|
| Down (−1) | Down continuation | Pause after decline | Support-like bounce |
| Unchanged (0) | Down after unchanged | Unchanged / unchanged | Up after unchanged |
| Up (+1) | Resistance-like rejection | Pause after rise | Up continuation |

Each trade belongs to one cell when both directions are known. There are no visit IDs. For example, down→unchanged followed by unchanged→up is **not** merged into one bounce.

Each cell stores count, current traded volume, and sums of all six differences. Dividing a difference sum by count gives its cell mean. Signed volume changes can cancel, so they remain explanatory measurements, not an automatic strength score.

Boundary rows are displayed separately and still contribute to the total activity distribution.

In [ ]:
# These are the six differences already calculated in step one.
# Listing their names lets us apply the same sum operation to each column.
difference_columns = ["time_backward", "time_forward", "price_backward", "price_forward", "volume_backward", "volume_forward"]
# Keep rows with BOTH directions known so each can be assigned to one of nine groups.
# pl.col selects a column, is_not_null checks availability, and & requires both checks.
# A direction of zero (unchanged) is valid and is retained.
complete = step1.filter(pl.col("arrival").is_not_null() & pl.col("departure").is_not_null())
# Retain rows missing either direction in a separate audit table; | means OR.
# The first arrival and final departure are unknown, not unchanged.
# Keeping them separately prevents silent loss of their count and volume.
boundary_rows = step1.filter(pl.col("arrival").is_null() | pl.col("departure").is_null())
# Group by exact integer price AND arrival AND departure.
# For example, all rows at one price with arrival=-1 and departure=+1 form one group.
# Integer price units avoid floating-point equality problems. agg calculates one row per group.
profile = complete.group_by("price_units", "arrival", "departure").agg(
    # Count the trades in this group. alias gives the result its readable column name.
    # This is a trade count, not a count of independent visits.
    pl.len().alias("trade_count"),
    # Sum CURRENT trade sizes in this group, not next-trade size or volume change.
    # Thus bounce-group volume means volume on trades immediately followed by an upward move.
    pl.col("volume").sum().alias("volume_sum"),
    # Build one sum expression per difference column; * expands the list into agg arguments.
    # Forward-time sum is the group's completed hold contribution, measured in seconds.
    # Backward time is kept separately: adding the two time sums would double-count intervals.
    # Price and volume differences are signed; positive and negative values can cancel.
    # Dividing each sum by trade_count yields that group's mean difference.
    *[pl.col(name).sum().alias(name + "_sum") for name in difference_columns],
# Close the aggregation, add a dollar-price column for display, then sort the rows.
# Grouping remains on price_units; dividing by 10,000 is presentation only.
# Sorting by price and both directions makes the table easier to inspect.
).with_columns((pl.col("price_units") / 10000).alias("price")).sort("price_units", "arrival", "departure")
# Display the first 18 aggregated rows only; the full profile remains in memory.
# This limits screen output, not the data used by subsequent calculations.
display(profile.head(18))
# Report how many original trade rows were classified and how many have unknown boundaries.
# DataFrame.height is its number of rows; these two counts should total N_EVENTS.
print("Classified rows:", complete.height, "boundary rows:", boundary_rows.height)
# Show the original event identities and known directions of the boundary rows.
# select chooses display columns; it does not modify or discard the saved boundary table.
display(boundary_rows.select("event", "price", "volume", "arrival", "departure"))

# Separately aggregate ALL selected trades by price, including the boundary rows.
# Activity totals need no next-trade outcome, so their count and volume are already known.
# This table describes overall trading concentration, not a particular reaction.
activity = raw.group_by("price_units").agg(pl.len().alias("total_trades"), pl.col("volume").sum().alias("total_volume")).sort("price_units")
# Rank prices by total trade count descending; break ties by lower price first.
# Select the first price_units value to choose one reproducible example for the matrix.
# "Busiest" here means most trades, not greatest volume or strongest support.
busiest = activity.sort(["total_trades", "price_units"], descending=[True, False])["price_units"][0]
# Explicitly create all nine (arrival, departure) pairs: each direction is -1, 0 or +1.
# The departure list repeats three times, once for each arrival direction.
# Int8 matches the direction-column types in profile so the join uses compatible keys.
grid = pl.DataFrame({"arrival": [-1,-1,-1,0,0,0,1,1,1], "departure": [-1,0,1]*3}).cast(pl.Int8)
# Filter the profile to the chosen price, then match rows by both directions.
# A LEFT join preserves all nine grid rows, even if a combination never occurred.
# An unobserved combination has null aggregate values after this join.
matrix = grid.join(profile.filter(pl.col("price_units") == busiest), on=["arrival", "departure"], how="left")
# Print the chosen price in dollars so the displayed matrix is clearly identified.
print("Nine cells at price:", busiest / 10000)
# Display directions plus count, volume and completed hold time for each of nine cells.
# Here null means NO MATCHING GROUP, so those three totals are correctly displayed as zero.
# This does not replace unknown directions or pending forward differences in the source.
# fill_null affects this display expression; the original matrix/profile remain unchanged.
display(matrix.select("arrival", "departure", pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0)))


## 4. Score each exact price first
Use the same equal-weight count, volume and time formula. Count and volume are normalized separately across prices for each role. Quick reversals get `tau / (tau + mean_forward_delay)`. Unknown boundaries are excluded from reaction scores; zero evidence gives score zero. Scores are relative strengths, not probabilities.

Only support (-1,+1) and resistance (+1,-1) define bands here. Unchanged and continuation observations remain in the original nine-cell table. The time reference tau remains 1 second; it is not a band-width parameter.


In [ ]:
# Define the four roles and their time interpretation explicitly.
roles = pl.DataFrame({
    "role": ["support", "resistance"],
    "arrival": [-1, 1],
    "departure": [1, -1],
    "time_preference": ["quick", "quick"],
}).with_columns(pl.col("arrival", "departure").cast(pl.Int8))

def score_bands(price_activity, reaction_profile, tau, weights):
    """Score every band/role, including empty cells. No neighbor or peak tests."""
    assert np.isfinite(tau) and tau > 0
    assert len(weights) == 3 and min(weights) >= 0 and np.isclose(sum(weights), 1)
    # Cross join creates both roles at every band, even when a cell is absent.
    table = price_activity.join(roles, how="cross").join(
        reaction_profile.select("band_id", "arrival", "departure", "trade_count", "volume_sum", "time_forward_sum"),
        on=["band_id", "arrival", "departure"], how="left",
    ).with_columns(
        # Missing group means zero observations/totals, not an unknown source direction.
        pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0),
    ).with_columns(
        # over('role') computes each role's maximum across bands, without merging roles.
        pl.col("trade_count").max().over("role").alias("max_count"),
        pl.col("volume_sum").max().over("role").alias("max_volume"),
        # No observations means no known mean delay; leave it null rather than set it to zero.
        pl.when(pl.col("trade_count") > 0).then(pl.col("time_forward_sum") / pl.col("trade_count"))
          .otherwise(None).alias("mean_delay_seconds"),
    ).with_columns(
        # If a role has no evidence anywhere, its count/volume components are zero.
        pl.when(pl.col("max_count") > 0).then(pl.col("trade_count") / pl.col("max_count"))
          .otherwise(0.0).alias("count_component"),
        pl.when(pl.col("max_volume") > 0).then(pl.col("volume_sum") / pl.col("max_volume"))
          .otherwise(0.0).alias("volume_component"),
        # Select QUICK for reversal roles and LONG for continuation roles.
        pl.when(pl.col("trade_count") == 0).then(0.0)
          .when(pl.col("time_preference") == "quick").then(tau / (tau + pl.col("mean_delay_seconds")))
          .otherwise(pl.col("mean_delay_seconds") / (tau + pl.col("mean_delay_seconds"))).alias("time_component"),
    ).with_columns(
        # Expose the contribution of each term so the final score is easy to audit.
        (weights[0] * pl.col("count_component")).alias("count_contribution"),
        (weights[1] * pl.col("volume_component")).alias("volume_contribution"),
        (weights[2] * pl.col("time_component")).alias("time_contribution"),
    ).with_columns(
        # An unobserved cell cannot earn a score from a time bonus.
        pl.when(pl.col("trade_count") > 0)
          .then(pl.col("count_contribution") + pl.col("volume_contribution") + pl.col("time_contribution"))
          .otherwise(0.0).alias("score"),
    )
    return table.sort(["role", "score", "band_id"], descending=[False, True, False])


In [ ]:
# Reuse the scorer with an exact price as the grouping key.
price_scores = score_bands(activity.rename({"price_units":"band_id"}),
    profile.rename({"price_units":"band_id"}), TAU_SECONDS,
    (WEIGHT_COUNT, WEIGHT_VOLUME, WEIGHT_TIME)).rename({"band_id":"price_units"})
price_scores = price_scores.with_columns((pl.col("price_units")/10000).alias("price"))
display(price_scores.select("role","price","trade_count","volume_sum","mean_delay_seconds",
    "count_component","volume_component","time_component","score").sort("role","price").head(30))


## 5. Locate peaks, plateaus and valleys
Sort all observed prices separately for each role, including zero-score prices. Consecutive exactly equal scores form one plateau. Compare each plateau with the previous and next plateau: higher than both is a peak; lower than both is a valley. Plateau representatives use the lower middle observed price when there are two middle choices.

An endpoint higher than its sole neighbor is an **edge peak**. Its outer boundary is open at the observed price-range edge. A wholly flat profile has no distinguishable peak and creates no bands. No rounding tolerance or smoothing is introduced.

Between consecutive peaks, choose the lowest-score plateau (the lower-priced one on an exact tie). Its representative price is the boundary. A boundary is drawn on both neighboring bands, but its evidence belongs only to the band on its right. The outer bands extend to the observed range edges. These are boundaries in the observed profile, not proof that all intermediate prices traded. Large unobserved gaps remain visible through member counts and width.


In [ ]:
def find_basins(prices, scores):
    """Deterministic one-dimensional partition; returns audit tables as records."""
    prices = list(map(int, prices)); scores = list(map(float, scores))
    assert len(prices) == len(scores) and len(prices) > 0
    assert all(a < b for a,b in zip(prices,prices[1:]))
    plateaus = []
    # A run of equal neighboring scores gets one representative, avoiding duplicate peaks.
    for i, value in enumerate(scores):
        if not plateaus or value != plateaus[-1]["score"]:
            plateaus.append(dict(start=i,end=i,score=value))
        else:
            plateaus[-1]["end"] = i
    for j,p in enumerate(plateaus):
        left = plateaus[j-1]["score"] if j else None
        right = plateaus[j+1]["score"] if j+1<len(plateaus) else None
        neighbors = [v for v in (left,right) if v is not None]
        p.update(plateau_id=j, representative=(p["start"]+p["end"])//2,
                 previous_score=left,next_score=right,
                 kind="peak" if neighbors and all(p["score"]>v for v in neighbors)
                      else "valley" if left is not None and right is not None and p["score"]<min(left,right)
                      else "flat" if not neighbors else "slope")
        p["price"] = prices[p["representative"]]/10000
    peaks = [p for p in plateaus if p["kind"]=="peak"]
    cuts = []
    for a,b in zip(peaks,peaks[1:]):
        between = plateaus[a["plateau_id"]+1:b["plateau_id"]]
        valley = min(between,key=lambda p:(p["score"],p["representative"]))
        cuts.append(valley["representative"])
    bands=[]; membership=[]
    for j,peak in enumerate(peaks):
        left = cuts[j-1] if j else 0
        right = cuts[j] if j<len(cuts) else len(prices)-1
        # A shared valley belongs to the right band; the last band includes the last price.
        stop = right if j<len(cuts) else len(prices)
        bands.append(dict(band_id=j,lower_units=prices[left],upper_units=prices[right],
            peak_units=prices[peak["representative"]],peak_score=peak["score"],
            lower_reason="valley" if j else "open range edge",
            upper_reason="valley" if j<len(cuts) else "open range edge",
            member_prices=stop-left))
        membership.extend(dict(price_units=prices[i],band_id=j) for i in range(left,stop))
    return plateaus,bands,membership

# Hand example from our discussion: the 100.04 valley divides the two peaks.
example_prices=[1000000+i*100 for i in range(8)]
example_scores=[.10,.35,.85,.60,.15,.50,.80,.20]
example_plateaus,example_bands,example_members=find_basins(example_prices,example_scores)
display(pl.DataFrame({"price":np.array(example_prices)/10000,"score":example_scores}))
display(pl.DataFrame(example_plateaus))
display(pl.DataFrame(example_bands))
display(pl.DataFrame(example_members).with_columns((pl.col("price_units")/10000).alias("price")))


## 6. Apply the boundaries and recompute band strength
The price score determines the shape of each role's bands. After partitioning, sum original count, current volume and forward-time evidence within each band, then recompute the same normalized score across bands of that role. Do not average price scores. The score at the peak is retained separately for comparison.

Support and resistance have independent partitions. Their evidence is conserved independently, not added across roles. The complete nine-cell price profile remains available; each band's score uses its own reversal cell only. The band start event is the earliest occurrence of an assigned member price.


In [ ]:
audit_rows=[]; definitions=[]; assignments=[]
for role in ("support","resistance"):
    ordered=price_scores.filter(pl.col("role")==role).sort("price_units")
    plateaus,bands,members=find_basins(ordered["price_units"],ordered["score"])
    audit_rows.extend(dict(role=role,**r) for r in plateaus)
    definitions.extend(dict(role=role,**r) for r in bands)
    assignments.extend(dict(role=role,**r) for r in members)
plateau_audit=pl.DataFrame(audit_rows)
boundaries=pl.DataFrame(definitions, schema_overrides={"band_id":pl.Int64})
membership=pl.DataFrame(assignments, schema_overrides={"price_units":pl.Int64,"band_id":pl.Int64})
# This sample has non-flat profiles; the flat-profile behavior is demonstrated and tested below.
assert boundaries.height > 0
contributions=membership.join(price_scores,on=["role","price_units"],how="left")
first_seen=raw.group_by("price_units").agg(pl.col("event").min().alias("first_event"))
band_scores=contributions.join(first_seen,on="price_units").group_by("role","band_id").agg(
    pl.col("trade_count").sum(),pl.col("volume_sum").sum(),pl.col("time_forward_sum").sum(),
    pl.col("first_event").min(),
).with_columns(
    (pl.col("trade_count")/pl.col("trade_count").max().over("role")).alias("count_component"),
    (pl.col("volume_sum")/pl.col("volume_sum").max().over("role")).alias("volume_component"),
    (pl.col("time_forward_sum")/pl.col("trade_count")).alias("mean_delay_seconds"),
).with_columns((TAU_SECONDS/(TAU_SECONDS+pl.col("mean_delay_seconds"))).alias("time_component"))
band_scores=band_scores.with_columns(
    ((pl.col("count_component")+pl.col("volume_component")+pl.col("time_component"))/3).alias("score")
).join(boundaries,on=["role","band_id"]).sort("role","lower_units")
# Full audit data is saved; displayed rows focus on the first few boundary decisions.
print("Plateau comparisons (full table: plateau_audit):")
display(plateau_audit.head(30))
print("Peaks and valleys:")
display(plateau_audit.filter(pl.col("kind").is_in(["peak","valley"])).head(30))
print("Boundary reasons and original member-price counts:")
display(boundaries.head(20))
print("Exact-price evidence assigned to bands:")
display(contributions.select("role","band_id","price","score","trade_count","volume_sum","time_forward_sum").head(30))
print("Recomputed band strengths:")
display(band_scores.head(20))
for name,table in dict(plateau_audit=plateau_audit,boundaries=boundaries,membership=membership,
                       contributions=contributions,band_scores=band_scores,price_scores=price_scores).items():
    table.write_parquet(RUN/(name+".parquet"))

# Follow one real interior support band from its boundary valleys to its source evidence.
example_band = boundaries.filter((pl.col("role")=="support") &
    (pl.col("lower_reason")=="valley") & (pl.col("upper_reason")=="valley")).head(1)
if example_band.height:
    chosen = example_band.row(0,named=True)
    print("Worked real band: support B", chosen["band_id"], "(upper boundary evidence belongs to the next band)")
    display(example_band)
    display(plateau_audit.filter((pl.col("role")=="support") &
        (pl.col("price")>=chosen["lower_units"]/10000) & (pl.col("price")<=chosen["upper_units"]/10000)))
    display(contributions.filter((pl.col("role")=="support") & (pl.col("band_id")==chosen["band_id"])))
    display(band_scores.filter((pl.col("role")=="support") & (pl.col("band_id")==chosen["band_id"])))


## 7. Replace the full partition with elevated score regions
Sections 4–6 above show notebook 05's original valley-to-valley partition for comparison. That partition covers the full observed price range, so its bands necessarily touch. The replay below instead keeps the connected region around each peak strictly above the higher of its two neighboring valley scores. The outer missing valley has score zero. Valley plateaus are left unassigned; narrow peaks may produce a single-price line. There is no gap multiplier or new numerical cutoff.

After every event, finalize the previous trade's reaction and update its count, volume and completed forward delay. Rebuild the two score profiles using observed prices only. Candidate evidence is summed inside each elevated region before normalization across that role's candidates. Existing bands keep their original bounds and base score for their current role period, so later evidence cannot move historical boundaries.

To avoid repeated overlapping discoveries, a new candidate is deferred if it overlaps an existing active or pending band of either role. Counts of deferred candidates are retained. Candidates outside the current price's valid support/resistance side are also deferred. Deferred candidates are reconsidered on subsequent events. This is a bounded research implementation: incremental evidence updates plus a price-profile scan per event, not a production checkpoint engine.


In [ ]:
def peak_cores(prices,scores):
    """Trim each valley basin to its elevated connected region around the peak."""
    plateaus,_,_=find_basins(prices,scores)
    peaks=[p for p in plateaus if p["kind"]=="peak"]
    cores=[]
    for peak in peaks:
        j=peak["plateau_id"]
        left=next((p for p in reversed(plateaus[:j]) if p["kind"]=="valley"),None)
        right=next((p for p in plateaus[j+1:] if p["kind"]=="valley"),None)
        floor=max(left["score"] if left else 0.,right["score"] if right else 0.)
        lo=hi=peak["representative"]
        while lo>0 and scores[lo-1]>floor:lo-=1
        while hi+1<len(prices) and scores[hi+1]>floor:hi+=1
        if scores[peak["representative"]]>floor:
            cores.append(dict(lo=lo,hi=hi,lower_units=int(prices[lo]),upper_units=int(prices[hi]),
                              peak_units=int(prices[peak["representative"]]),valley_floor=floor))
    return cores

# The same hand profile now leaves the 100.04 valley out of both bands.
hand_cores=peak_cores(example_prices,example_scores)
display(pl.DataFrame(hand_cores))
assert hand_cores[0]["upper_units"]<hand_cores[1]["lower_units"]


## 8. Chronological lifecycle: discover → break → wait → opposite-side retest
A support breaks on the first trade strictly below its lower bound. It becomes pending resistance, with no active strength. Resistance is symmetric. A pending resistance activates only when price moves from below the band into it; pending support requires entry from above. A gap across the entire band does not count as a retest. Entry confirms the retest for this method; no additional rejection or hold duration is required.

At retest, start a new role period with the same boundaries, a fresh base score from that role's currently completed evidence, and age zero. Never copy support evidence into resistance. The old period remains as history. New non-overlapping candidates continue to be discovered from later events. A zone has one lifecycle at a time; pending zones block overlapping duplicate discoveries of either role.

Each active period uses `B + (1-B)*age/session_elapsed`. Breaks end that period. The time rule still favors early periods and rewards untested survival; it is a heuristic. Historical display thresholds use the last intact score, while pending bands have no active score.


In [ ]:
def replay_evolving(frame):
    # Future coordinates are never used: the dictionary contains observed prices only.
    evidence={};first_seen={};periods=[];live=[];pending=[];transitions=[]
    deferred={"overlap":0,"wrong_side":0};last_candidates=[]
    rows=frame.select("event","time_us","price_units","volume").to_dicts()
    for i,row in enumerate(rows):
        event=int(row["event"]);now=int(row["time_us"]);price=int(row["price_units"])
        evidence.setdefault(price,np.zeros((2,3),dtype=float));first_seen.setdefault(price,event)
        # Event i reveals the departure of i-1; event zero has no arrival.
        if i>=2:
            previous=rows[i-1];pp=int(previous["price_units"])
            arrival=np.sign(pp-int(rows[i-2]["price_units"]));departure=np.sign(price-pp)
            role_index=0 if (arrival,departure)==(-1,1) else 1 if (arrival,departure)==(1,-1) else None
            if role_index is not None:
                evidence[pp][role_index]+=np.array([1.,float(previous["volume"]),(now-int(previous["time_us"]))/1e6])
        if event<DISCOVERY_EVENT:continue
        prices=np.array(sorted(evidence),dtype=np.int64)
        values=np.stack([evidence[int(p)] for p in prices])
        candidates=[];normalizers={}
        for ri,role in enumerate(("support","resistance")):
            counts,volumes,delays=values[:,ri,:].T
            mean=np.divide(delays,counts,out=np.zeros_like(delays),where=counts>0)
            scores=np.where(counts>0,(counts/max(counts.max(),1)+volumes/max(volumes.max(),1)+TAU_SECONDS/(TAU_SECONDS+mean))/3,0.)
            cores=peak_cores(prices,scores)
            for core in cores:
                totals=values[core["lo"]:core["hi"]+1,ri,:].sum(axis=0)
                core.update(role=role,trade_count=float(totals[0]),volume_sum=float(totals[1]),time_sum=float(totals[2]),
                            origin_event=min(first_seen[int(p)] for p in prices[core["lo"]:core["hi"]+1]))
            normalizers[role]=(max([c["trade_count"] for c in cores]+[1.]),max([c["volume_sum"] for c in cores]+[1.]))
            candidates.extend(cores)
        def base_score(role,lower,upper):
            ri=0 if role=="support" else 1
            count,volume,delay=values[(prices>=lower)&(prices<=upper),ri,:].sum(axis=0)
            if count==0:return 0.
            nc,nv=normalizers[role]
            # Historical bounds can differ from current candidates; clamp relative components.
            return float((min(count/nc,1.)+min(volume/nv,1.)+TAU_SECONDS/(TAU_SECONDS+delay/count))/3)
        def activate(role,lower,upper,origin,cause,parent=None):
            record=dict(period_id=len(periods),role=role,lower_units=lower,upper_units=upper,
                        origin_event=origin,start_event=event,start_us=now,base_score=base_score(role,lower,upper),
                        end_event=None,end_us=None,cause=cause,parent_period=parent)
            periods.append(record);live.append(record)
            transitions.append(dict(event=event,period_id=record["period_id"],action=cause,role=role))
        # Break first; the same event cannot count as a later retest.
        for record in list(live):
            breaks=price<record["lower_units"] if record["role"]=="support" else price>record["upper_units"]
            if breaks:
                record.update(end_event=event,end_us=now);live.remove(record)
                opposite="resistance" if record["role"]=="support" else "support"
                pending.append(dict(role=opposite,lower_units=record["lower_units"],upper_units=record["upper_units"],
                                    origin_event=event,broken_event=event,parent_period=record["period_id"]))
                transitions.append(dict(event=event,period_id=record["period_id"],action="broken; waiting for "+opposite,role=record["role"]))
        for wait in list(pending):
            if event<=wait["broken_event"] or i==0:continue
            prev=int(rows[i-1]["price_units"])
            inside=wait["lower_units"]<=price<=wait["upper_units"]
            approach=prev<wait["lower_units"] if wait["role"]=="resistance" else prev>wait["upper_units"]
            if inside and approach:
                pending.remove(wait)
                activate(wait["role"],wait["lower_units"],wait["upper_units"],event,"opposite-side retest",wait["parent_period"])
        # Stronger candidates get first consideration when candidate regions overlap old zones.
        candidates.sort(key=lambda c:base_score(c["role"],c["lower_units"],c["upper_units"]),reverse=True)
        for c in candidates:
            valid=price>=c["lower_units"] if c["role"]=="support" else price<=c["upper_units"]
            if not valid:deferred["wrong_side"]+=1;continue
            occupied=any(z["lower_units"]<=c["upper_units"] and c["lower_units"]<=z["upper_units"] for z in live+pending)
            if occupied:deferred["overlap"]+=1;continue
            activate(c["role"],c["lower_units"],c["upper_units"],c["origin_event"],"discovered")
        last_candidates=candidates
    return dict(periods=periods,pending=pending,transitions=transitions,deferred=deferred,candidates=last_candidates)

# Source is only 2,000 events; this replay records transitions, not an event-by-band matrix.
replay=replay_evolving(source)
print("Role periods:",len(replay["periods"]),"pending retests:",len(replay["pending"]),"deferred candidate observations:",replay["deferred"])
display(pl.DataFrame(replay["transitions"]).head(25))
display(pl.DataFrame(replay["candidates"]).head(15))


## 9. As-of state and intermediate time calculation
The precomputed chronological history is masked at the chosen event: later discoveries, breaks and retests are not visible. Active periods run to the cutoff; broken periods end at their break. The age of a new role starts at its retest, not at the old role's origin. The chart has no legend. Labels show role, ID, score and status.

Faint dotted segments from a discovered band's first member-price occurrence to its discovery are retrospective context only. Retested roles start exactly at the retest; the old role's history is retained separately. Thresholds filter current active scores or last intact historical scores. No future price trace or future level is displayed beyond the as-of event.


In [ ]:
def state_at(event,history=replay):
    now=int(source["time_us"][event-1]);elapsed=(now-SESSION_OPEN_US)/1e6
    records=[]
    for p in history["periods"]:
        if p["start_event"]>event:continue
        broken=p["end_event"] is not None and p["end_event"]<=event
        end=p["end_event"] if broken else event
        end_us=p["end_us"] if broken else now
        age=(end_us-p["start_us"])/1e6
        fraction=age/elapsed if elapsed>0 else 0.
        strength=0. if broken else p["base_score"]+(1-p["base_score"])*fraction
        historic_elapsed=(end_us-SESSION_OPEN_US)/1e6
        display_score=p["base_score"]+(1-p["base_score"])*age/historic_elapsed if historic_elapsed>0 else p["base_score"]
        records.append(dict(period_id=p["period_id"],role=p["role"],lower_units=p["lower_units"],upper_units=p["upper_units"],
            origin_event=p["origin_event"],start_event=p["start_event"],draw_end_event=end,
            broken_event=p["end_event"] if broken else None,status="broken" if broken else "active",
            cause=p["cause"],base_score=p["base_score"],session_elapsed_seconds=elapsed,intact_seconds=age,
            time_fraction=fraction,time_bonus=0. if broken else (1-p["base_score"])*fraction,
            strength=strength,display_score=display_score))
    return pl.DataFrame(records)

for cutoff in [200,300,593,1000,2000]:
    t=state_at(cutoff)
    print("As of",cutoff)
    display(t.group_by("role","status").len())
display(state_at(593).select("period_id","role","cause","start_event","broken_event","base_score",
                           "session_elapsed_seconds","intact_seconds","time_fraction","time_bonus","strength").head(25))


In [ ]:
import ipywidgets as widgets
colors={"support":"#16803c","resistance":"#c53030"};chart_state={}
def draw_time_bands(events,support,resistance):
    state=state_at(events);observed=source.head(events)
    thresholds={"support":support,"resistance":resistance}
    selected=state.join(pl.DataFrame({"role":list(thresholds),"threshold":list(thresholds.values())}),on="role").filter(pl.col("display_score")>=pl.col("threshold"))
    print("As of event",events,"| continuous discovery | X = break; new role starts only at opposite-side retest")
    display(state.group_by("role","status").len().sort("role","status"))
    fig,ax=plt.subplots(figsize=(18,10));fig.subplots_adjust(right=.80)
    ax.plot(observed["event"],observed["price"],color="black",lw=.8)
    for row in selected.iter_rows(named=True):
        color=colors[row["role"]];lo=row["lower_units"]/10000;hi=row["upper_units"]/10000
        start=row["start_event"];end=row["draw_end_event"]
        if row["origin_event"]<start:
            ax.hlines([lo,hi],row["origin_event"],start,color=color,lw=.5,alpha=.3,linestyles=":")
        ax.fill_between([start,end],lo,hi,color=color,alpha=.10)
        ax.hlines([lo,hi],start,end,color=color,lw=.8,alpha=.7,linestyles="--" if row["role"]=="resistance" else "-")
        if row["status"]=="broken":ax.scatter([end],[(lo+hi)/2],marker="x",s=12,color=color)
    # Label active bands only; historical details remain in the table, avoiding long crossing leaders.
    active=selected.filter(pl.col("status")=="active").with_columns(((pl.col("lower_units")+pl.col("upper_units"))/20000).alias("center")).sort("center")
    low,high=ax.get_ylim();ys=np.linspace(low+.03*(high-low),high-.03*(high-low),active.height) if active.height>1 else active["center"].to_numpy()
    for row,y in zip(active.iter_rows(named=True),ys):
        ax.annotate(f"{row['role']} #{row['period_id']} | {row['strength']:.3f}",xy=(events,row["center"]),xytext=(1.01,y),
                    textcoords=ax.get_yaxis_transform(),color=colors[row["role"]],fontsize=8,annotation_clip=False,
                    arrowprops=dict(arrowstyle="-",color=colors[row["role"]],lw=.4,relpos=(0,.5)))
    ax.set(xlim=(1,events),xlabel="As-of event",ylabel="Price ($)",title="Evolving support / resistance — breaks and opposite-side retests")
    ax.grid(alpha=.12)
    fig.savefig(RUN/"unbroken_age_bands.png",dpi=140,bbox_inches="tight");plt.show();plt.close(fig)
    display(state.select("period_id","role","cause","origin_event","start_event","broken_event","status","base_score","intact_seconds","time_bonus","strength").tail(30))
    state.write_parquet(RUN/"current_time_state.parquet")
    chart_state.update(state=state,selected=selected,event=events,renders=chart_state.get("renders",0)+1)
controls={"events":widgets.IntSlider(value=593,min=DISCOVERY_EVENT,max=source.height,description="As-of event",continuous_update=False,layout=widgets.Layout(width="900px"))}
controls.update({r:widgets.FloatSlider(value=.5,min=0,max=1,step=.01,description=r.title(),continuous_update=False,layout=widgets.Layout(width="450px")) for r in colors})
output=widgets.interactive_output(draw_time_bands,controls)
display(widgets.VBox([controls["events"],widgets.HBox([controls["support"],controls["resistance"]]),output]))


## 10. Check prefix invariance, retest transitions and time state
Validate against shorter replays. Check each role transition against the preceding price, entry price, previous role, and prior break; check frozen boundaries, reset age, strict breaks and bounds. These tests validate implementation, not predictive effectiveness.


In [ ]:
short=replay_evolving(source.head(593))
assert state_at(593).equals(state_at(593,short))
assert any(p["start_event"]>DISCOVERY_EVENT and p["cause"]=="discovered" for p in replay["periods"])
flips=[p for p in replay["periods"] if p["cause"]=="opposite-side retest"]
assert flips,"Sample should exercise retests"
for p in replay["periods"]:
    if p["end_event"] is not None:
        price=int(source["price_units"][p["end_event"]-1])
        assert price<p["lower_units"] if p["role"]=="support" else price>p["upper_units"]
    if p["parent_period"] is not None:
        parent=replay["periods"][p["parent_period"]]
        assert parent["end_event"]<p["start_event"] and parent["role"]!=p["role"]
        assert (p["lower_units"],p["upper_units"])==(parent["lower_units"],parent["upper_units"])
        prev=int(source["price_units"][p["start_event"]-2]);price=int(source["price_units"][p["start_event"]-1])
        assert p["lower_units"]<=price<=p["upper_units"]
        assert prev<p["lower_units"] if p["role"]=="resistance" else prev>p["upper_units"]
        born=state_at(p["start_event"]).filter(pl.col("period_id")==p["period_id"])
        assert born["intact_seconds"][0]==0 and born["time_bonus"][0]==0
for cutoff in [200,593,1000,2000]:
    state=state_at(cutoff)
    assert (state["draw_end_event"]<=cutoff).all()
    assert state["strength"].min()>=0 and state["strength"].max()<=1
    assert state.filter(pl.col("status")=="broken")["strength"].sum()==0
manifest=dict(method="evolving-peak-cores-retest",source_sha256=source_hash,events=source.height,
    periods=len(replay["periods"]),retests=len(flips),deferred=replay["deferred"],checks="passed",
    git_commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip())
(RUN/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
(RUN/"lifecycle.json").write_text(json.dumps(replay),encoding="utf-8")
(RUN/"source_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
print(json.dumps(manifest,indent=2))
